In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# Load data - correct path
df = pd.read_csv('listings.csv')

print(f"Loaded {len(df)} listings")

# Clean data
df = df.dropna(subset=['price', 'latitude', 'longitude', 'neighbourhood', 'room_type'])
df = df[df['price'] > 0]
df = df[df['price'] < 1000]

print(f"After cleaning: {len(df)} listings")

# Feature engineering
le_neighborhood = LabelEncoder()
le_room_type = LabelEncoder()

df['neighborhood_code'] = le_neighborhood.fit_transform(df['neighbourhood'].astype(str))
df['room_type_code'] = le_room_type.fit_transform(df['room_type'].astype(str))

# Distance to city center
city_center_lat = 51.5074
city_center_lon = -0.1278
df['distance_to_center'] = np.sqrt(
    (df['latitude'] - city_center_lat)**2 + 
    (df['longitude'] - city_center_lon)**2
)

# Host popularity
host_listing_count = df.groupby('host_id')['id'].count().reset_index(name='host_listing_count')
df = df.merge(host_listing_count, on='host_id')

# Select features
features = ['latitude', 'longitude', 'neighborhood_code', 'room_type_code', 
            'distance_to_center', 'host_listing_count']
X = df[features]
y = df['price']

print(f"Features: {features}")
print(f"X shape: {X.shape}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numeric features
scaler = StandardScaler()
numeric_cols = ['latitude', 'longitude', 'distance_to_center', 'host_listing_count']
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Train models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    
    print(f"\n{name}")
    print(f"  MAE: £{mae:.2f}")
    print(f"  RMSE: £{rmse:.2f}")
    print(f"  R² Score: {r2:.4f}")

# Feature importance (Random Forest)
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importance:")
print(feature_importance)

# Save the best model
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le_neighborhood, '../models/le_neighborhood.pkl')
joblib.dump(le_room_type, '../models/le_room_type.pkl')

print("\n✅ Models saved successfully!")

Loaded 69351 listings
After cleaning: 67911 listings
Features: ['latitude', 'longitude', 'neighborhood_code', 'room_type_code', 'distance_to_center', 'host_listing_count']
X shape: (67911, 6)

Linear Regression
  MAE: £72.87
  RMSE: £117.40
  R² Score: 0.2252

Random Forest
  MAE: £59.80
  RMSE: £100.58
  R² Score: 0.4314

Feature Importance:
              feature  importance
0            latitude    0.229151
4  distance_to_center    0.202416
3      room_type_code    0.195900
1           longitude    0.191815
5  host_listing_count    0.161844
2   neighborhood_code    0.018874

✅ Models saved successfully!
